In [1]:
%pip install torch torch_geometric thefuzz python-Levenshtein networkx matplotlib pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import networkx as nx
import torch
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from thefuzz import process, fuzz
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
import pickle

c:\Users\kanis\Desktop\SkillBridge\skillbridge-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.makedirs('outputs', exist_ok=True)

In [4]:
skills         = pd.read_csv(f'Data/esco/skills_en.csv')
occupations    = pd.read_csv(f'Data/esco/occupations_en.csv')
skill_rels     = pd.read_csv(f'Data/esco/skillSkillRelations_en.csv')
occ_skill_rels = pd.read_csv(f'Data/esco/occupationSkillRelations_en.csv')
 
print(f"ESCO Skills:              {len(skills):>6}")
print(f"ESCO Occupations:         {len(occupations):>6}")
print(f"Skill-Skill relations:    {len(skill_rels):>6}")
print(f"Occ-Skill relations:      {len(occ_skill_rels):>6}")

ESCO Skills:               13960
ESCO Occupations:           3043
Skill-Skill relations:      5818
Occ-Skill relations:      126051


In [5]:
onet_sw  = pd.read_csv(f'Data/onet/Software Skills.txt', sep='\t')
onet_ess = pd.read_csv(f'Data/onet/Essential Skills.txt', sep='\t')
onet_occ = pd.read_csv(f'Data/onet/Occupation Data.txt', sep='\t')
 
onet_skill_names = list(set(
    onet_sw['Element Name'].unique().tolist() +
    onet_ess['Element Name'].unique().tolist()
))
esco_skill_labels = skills['preferredLabel'].dropna().tolist()
 
matches = []
for onet_skill in onet_skill_names:
    best_match, score = process.extractOne(
        onet_skill, esco_skill_labels, scorer=fuzz.token_sort_ratio
    )
    print(f"O*NET: {onet_skill} | ESCO: {best_match} | Score: {score}")
    matches.append({'onet': onet_skill, 'esco': best_match, 'score': score})
 
overlap_df = pd.DataFrame(matches)
high_conf = overlap_df[overlap_df['score'] >= 80]
print(f"\nO*NET unique skill names:        {len(onet_skill_names)}")
print(f"ESCO skill labels:               {len(esco_skill_labels)}")
print(f"High-confidence overlaps (>=80): {len(high_conf)} / {len(onet_skill_names)}")

O*NET: Word processing software | ESCO: use word processing software | Score: 92
O*NET: Device drivers or system software | ESCO: customise software for drive system | Score: 79
O*NET: Eye tracking software | ESCO: trading software | Score: 81
O*NET: File versioning software | ESCO: plan software testing | Score: 71
O*NET: WAN switching software and firmware | ESCO: perform software unit testing | Score: 62
O*NET: Object or component oriented development software | ESCO: oversee development of software | Score: 68
O*NET: Storage media loading software | ESCO: use media software | Score: 71
O*NET: Risk management data and analysis software | ESCO: apply risk management in sports | Score: 68
O*NET: Voice recognition software | ESCO: use presentation software | Score: 71
O*NET: Clustering software | ESCO: authoring software | Score: 81
O*NET: Point of sale POS software | ESCO: vessel points of sail | Score: 68
O*NET: Configuration management software | ESCO: tools for software configurati

In [6]:
cs_occs     = occupations[occupations['iscoGroup'].astype(str).str.startswith('25')]
cs_occ_uris = set(cs_occs['conceptUri'])
 
# Skills connected to CS occupations
cs_skill_uris = set(
    occ_skill_rels[occ_skill_rels['occupationUri'].isin(cs_occ_uris)]['skillUri']
)
 
# Build graph
G = nx.MultiDiGraph()
skill_label_map = dict(zip(skills['conceptUri'], skills['preferredLabel']))
occ_label_map   = dict(zip(occupations['conceptUri'], occupations['preferredLabel']))
 
for uri in cs_skill_uris:
    G.add_node(uri, node_type='skill', label=skill_label_map.get(uri, uri), source='esco')
for uri in cs_occ_uris:
    G.add_node(uri, node_type='occupation', label=occ_label_map.get(uri, uri), source='esco')
 
# ESCO edges
for _, row in occ_skill_rels.iterrows():
    if row['occupationUri'] in cs_occ_uris and row['skillUri'] in cs_skill_uris:
        G.add_edge(row['skillUri'], row['occupationUri'], edge_type=row['relationType'])
 
for _, row in skill_rels.iterrows():
    if row['originalSkillUri'] in cs_skill_uris and row['relatedSkillUri'] in cs_skill_uris:
        G.add_edge(row['originalSkillUri'], row['relatedSkillUri'],
                   edge_type=row['relationType'])
 
# O*NET edges (software skills for CS occupations, SOC 15-x)
onet_cs_codes = set(onet_occ[onet_occ['O*NET-SOC Code'].str.startswith('15-')]['O*NET-SOC Code'])
onet_sw_cs    = onet_sw[onet_sw['O*NET-SOC Code'].isin(onet_cs_codes)]
 
for name in onet_sw_cs['Element Name'].unique():
    G.add_node(f'onet_skill_{name}', node_type='skill', label=name, source='onet')
for code in onet_cs_codes:
    title = onet_occ[onet_occ['O*NET-SOC Code'] == code]['Title'].values
    G.add_node(f'onet_occ_{code}', node_type='onet_occupation',
               label=title[0] if len(title) > 0 else code, source='onet')
for _, row in onet_sw_cs.iterrows():
    G.add_edge(f"onet_skill_{row['Element Name']}",
               f"onet_occ_{row['O*NET-SOC Code']}",
               edge_type='onet_software', source='onet')
 
# Remove isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))
 
skill_count = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'skill')
occ_count   = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'occupation')
edge_dist   = Counter(d['edge_type'] for _, _, d in G.edges(data=True))
 
print(f"\nNetworkX Graph:")
print(f"  Skill nodes:       {skill_count}")
print(f"  Occupation nodes:  {occ_count}")
print(f"  Total nodes:       {G.number_of_nodes()}")
print(f"  Total edges:       {G.number_of_edges()}")
print(f"  Edge types:        {dict(edge_dist)}")


NetworkX Graph:
  Skill nodes:       979
  Occupation nodes:  75
  Total nodes:       1090
  Total edges:       11431
  Edge types:        {'optional': 3369, 'essential': 1845, 'onet_software': 6217}


In [7]:
import matplotlib.pyplot as plt
import networkx as nx

# Pick 3 ESCO occupations
if 'G' not in globals():
    raise NameError("G is not defined. Run the graph construction cell first.")

sample_occ = [
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'occupation' and not n.startswith('onet_occ_')
][:3]

print("Selected occupations:")
for n in sample_occ:
    print(f"  {G.nodes[n]['label']}")

# Get skills connected to these occupations
connected_skills = []
for occ in sample_occ:
    preds = [p for p in G.predecessors(occ)
             if G.nodes[p]['node_type'] == 'skill'][:6]
    connected_skills.extend(preds)

sample_nodes = list(set(connected_skills)) + sample_occ
for n in sample_nodes:
    print(f"{G.nodes[n]['label'][:30]} — {G.nodes[n]['node_type']}")

print(f"\nTotal nodes in sample: {len(sample_nodes)}")
print(f"Skills: {sum(1 for n in sample_nodes if G.nodes[n]['node_type'] == 'skill')}")
print(f"Occupations: {sum(1 for n in sample_nodes if G.nodes[n]['node_type'] == 'occupation')}")

small  = G.subgraph(sample_nodes)
labels = {n: G.nodes[n].get('label', '')[:15] for n in sample_nodes}
colors = [
    '#E74C3C' if G.nodes[n]['node_type'] in {'occupation', 'onet_occupation'} else '#4A90D9'
    for n in sample_nodes
]

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(small, seed=42, k=3)
# ensure color list aligns with drawn nodes
nx.draw(small, pos, nodelist=sample_nodes, labels=labels, node_color=colors, node_size=2000,
        font_size=8, arrows=True, edge_color='#aaa', width=1.5)
plt.title('CS Subgraph — Blue=Skill, Red=Occupation', fontsize=12)
plt.savefig('outputs/cs_subgraph_sample.png', dpi=150)
plt.show()
print("Saved.")

Selected occupations:
  ICT usability tester
  ethical hacker
  cloud software developer
use experience map — skill
levels of software testing — skill
ICT network security risks — skill
decentralized application fram — skill
Java (computer programming) — skill
building systems monitoring te — skill
application usability — skill
computer forensics — skill
software frameworks — skill
information security strategy — skill
computer programming — skill
cloud security and compliance — skill
human-computer interaction — skill
JavaScript — skill
computer engineering — skill
ethical hacking principles — skill
behavioural science — skill
test for emotional patterns — skill
ICT usability tester — occupation
ethical hacker — occupation
cloud software developer — occupation

Total nodes in sample: 21
Skills: 18
Occupations: 3
Saved.


C:\Users\kanis\AppData\Local\Temp\ipykernel_17548\1330904601.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
import matplotlib.pyplot as plt
import networkx as nx

# Only pick ESCO occupation nodes (not O*NET ones)
sample_occ = [
    n for n, d in G.nodes(data=True) 
    if d['node_type'] == 'occupation' and not n.startswith('onet_occ_')
][:3]

connected_skills = []
for occ in sample_occ:
    preds = list(G.predecessors(occ))[:6]
    connected_skills.extend(preds)

sample_nodes = list(set(connected_skills)) + sample_occ
small = G.subgraph(sample_nodes)

labels = {n: G.nodes[n].get('label', '')[:15] for n in sample_nodes}
colors = [
    '#E74C3C' if G.nodes[n]['node_type'] in {'occupation', 'onet_occupation'} else '#4A90D9'
    for n in sample_nodes
]

plt.figure(figsize=(16, 12))
pos = nx.spring_layout(small, seed=42, k=3)
# ensure color list aligns with drawn nodes
nx.draw(small, pos, nodelist=sample_nodes, labels=labels, node_color=colors, node_size=2000,
        font_size=8, arrows=True, edge_color='#aaa', width=1.5)
plt.title('CS Subgraph — Blue=Skill, Red=Occupation', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/cs_subgraph_sample.png', dpi=150)
plt.show()

C:\Users\kanis\AppData\Local\Temp\ipykernel_17548\82999146.py:30: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\kanis\AppData\Local\Temp\ipykernel_17548\82999146.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Combine the ESCO and O*NET skill tables (robust if earlier cells not run in order)
# Ensure source dataframes exist or build minimal versions from available frames
if 'cs_skills_df' not in globals():
    if 'skills' in globals() and 'cs_skill_uris' in globals():
        cs_skills_df = skills[skills['conceptUri'].isin(cs_skill_uris)][['conceptUri', 'preferredLabel', 'skillType']].reset_index(drop=True)
        cs_skills_df['source'] = 'esco'
    else:
        cs_skills_df = pd.DataFrame(columns=['conceptUri','preferredLabel','skillType','source'])

if 'onet_skills_df' not in globals():
    if 'onet_sw_cs' in globals():
        onet_unique = list(onet_sw_cs['Element Name'].unique())
        onet_skills_df = pd.DataFrame({
            'conceptUri': [f'onet_{i}' for i in range(len(onet_unique))],
            'preferredLabel': onet_unique,
            'skillType': ['onet_software'] * len(onet_unique),
            'source': ['onet'] * len(onet_unique)
        })
    else:
        onet_skills_df = pd.DataFrame(columns=['conceptUri','preferredLabel','skillType','source'])

combined = pd.concat([cs_skills_df, onet_skills_df], ignore_index=True).sort_values('preferredLabel')
combined.index = range(1, len(combined) + 1)

In [ ]:
def assign_category_smart(skill_name):
    s = skill_name.lower()

    # --- Programming Languages ---
    prog_keywords = [
        'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'ruby',
        'swift', 'kotlin', 'golang', 'rust', 'scala', 'php', 'perl', 'matlab',
        'bash', 'shell', 'haskell', 'erlang', 'cobol', 'fortran', 'lua',
        'assembly', 'scripting', 'programming', 'object-oriented', 'sas language',
        'dart', 'groovy', 'lisp', 'prolog', 'abap', 'vba', 'powershell',
        'ajax', 'apl', 'asp.net', 'coffeescript', 'css', 'less', 'sass',
        'linq', 'mdx', 'n1ql', 'sparql', 'vbscript', 'visual basic',
        'solidity', 'vyper', 'objective-c', 'html', 'xml', 'xquery',
        'r language', 'spark', 'staf', 'jsss', 'algorithms',
        ' r ', 'ldap', 'utilise regular expressions'
    ]

    # --- Databases & Data ---
    data_keywords = [
        'sql', 'database', 'nosql', 'mongodb', 'redis', 'cassandra',
        'elasticsearch', 'data model', 'data warehouse', 'etl', 'data mining',
        'big data', 'data lake', 'data science', 'data engineer', 'data quality',
        'data process', 'data collect', 'data storage', 'data retriev',
        'data migrat', 'data classif', 'data integr', 'data govern',
        'data protect', 'data manag', 'data analys', 'data extract',
        'data set', 'data sample', 'data cleans', 'data preserv',
        'data security', 'data ethics', 'data visualis',
        'business intelligence', 'analytics', 'statistical', 'statistics',
        'machine learning', 'deep learning', 'neural', 'natural language processing',
        'computer vision', 'artificial intelligence', 'predictive model',
        'recommender', 'dimensionality', 'normalise data', 'online analytical',
        'information retrieval', 'information management', 'information system',
        'information structure', 'information categor', 'information extract',
        'information governance', 'information security',
        'distributed computing', 'quantum computing', 'computational',
        'mathematical model', 'mathematical physics', 'mathematics',
        'quantitative', 'blockchain', 'cryptocurrency', 'distributed ledger',
        'decentralised', 'decentralized', 'smart contract', 'augmented reality',
        'scientific computing', 'db2', 'hadoop', 'informix', 'datastage',
        'informatica', 'marklogic', 'objectstore', 'pentaho', 'triplestore',
        'ca datacom', 'sap data', 'sap r3', 'oracle warehouse',
        'qlikview', 'risk management data', 'decision support',
        'deliver visual presentation of data', 'interpret current data',
        'make data-driven', 'migrate existing data', 'process data',
        'unstructured data', 'image recognition', 'empirical analysis',
        'investment analysis', 'operational research', 'game theory',
        'synthesise information', 'synthesise research', 'trigonometry',
        'real-time computing', 'embedded systems', 'statistics',
        'mobile device software', 'software frameworks',
        'decentralized application', 'cognitive computing'
    ]

    # --- Frameworks & Tools ---
    tools_keywords = [
        'software development', 'software design', 'software architect',
        'software test', 'software engineer', 'software quality',
        'software configur', 'software maintenance', 'software recovery',
        'software release', 'software metric', 'software anomal',
        'software component', 'software interact', 'software unit',
        'software process', 'software specif', 'software framework',
        'web development', 'web design', 'web page', 'web platform',
        'front-end', 'frontend', 'user interface', 'user experience',
        'ux', 'wireframe', 'prototype', 'design pattern', 'uml',
        'agile', 'scrum', 'devops', 'ci/cd', 'version control', 'git',
        'docker', 'kubernetes', 'ansible', 'vagrant', 'octopus',
        'apache maven', 'apache tomcat', 'wildfly', 'ibm websphere',
        'codenvy', 'xcode', 'ios', 'android', 'eclipse',
        'react', 'angular', 'vue', 'django', 'flask', 'spring', 'node',
        'wordpress', 'drupal', 'joomla', 'webcms', 'kdevelop',
        'tensorflow', 'pytorch', 'keras',
        'adobe', 'gimp', 'sketchbook', 'synfig',
        '3d model', '3d light', '3d textur', '3d image', 'render 3d',
        'cad software', 'cam software', 'computer aided design',
        'computer imaging', 'graphics', 'animation', 'visual design',
        'debug', 'code review', 'code exploit', 'reverse engineer',
        'test suite', 'test automat', 'integration test', 'unit test',
        'document management', 'content management', 'content workflow',
        'project management', 'configuration management',
        'enterprise application', 'enterprise system',
        'presentation software', 'spreadsheet', 'office suite',
        'video creation', 'video editing', 'music', 'sound edit',
        'flowchart', 'semantic tree', 'design sketch',
        'iterative development', 'waterfall', 'spiral development',
        'systems development life', 'application development',
        'mobile development', 'game engine', 'game test',
        'reporting software', 'authoring software', 'file versioning',
        'compiler', 'device driver', 'ict workflow', 'ict test',
        'ict device', 'implement smart', 'integrate system',
        'service-oriented', 'solution deployment', 'system design',
        'information architect', 'interaction model', 'component interface',
        'enterprise architect', 'model based system',
        'map creation', 'medical software', 'industrial control',
        'manufacturing execution', 'point of sale', 'calendar and sched',
        'instant messaging', 'video conferenc', 'electronic mail',
        'helpdesk', 'desktop publishing', 'optical character',
        'data compress', 'gateway software', 'filesystem',
        'expert system', 'platform interconnect', 'pattern design',
        'categorization or classification', 'human-computer',
        'microsoft access', 'microsoft visio', 'blackberry',
        'process mapping', 'business process model', 'create flowchart',
        'task algorithmis', 'signal processing', 'field-programmable',
        'image formation', 'interfacing', 'technical documentation',
        'blueprints', 'technical drawings', 'trading software',
        'screen reader', 'mobile location', 'multi-media educational',
        'computer based training', 'crm software',
        'customer relationship management', 'accounting software',
        'administration software', 'human resources software',
        'sales and marketing software', 'financial analysis software',
        'time accounting', 'analytical or scientific software',
        'access software', 'word processing', 'bridge software',
        'clustering software', 'compliance software', 'storage media',
        'transaction security', 'procedure management',
        'requirements analysis', 'object or component', 'object oriented data',
        'open source model', 'operate open source', 'program testing',
        'integrated development environment', 'development environment',
        'application interface', 'application-specific interface',
        'design application', 'design process', 'design thinking',
        'align software', 'analyse ict system', 'adjust ict',
        'administer ict', 'attend to ict', 'business ict',
        'configure ict', 'deploy ict', 'innovate in ict',
        'maintain ict system', 'manage ict data arch', 'manage ict project',
        'manage ict semantic', 'manage changes in ict', 'manage system test',
        'optimise choice of ict', 'propose ict', 'provide ict system training',
        'solve ict system', 'support ict system', 'apply ict system',
        'oversee development', 'replicate customer software',
        'application usability', 'usability engineering',
        'incremental development', 'prototyping development',
        'content development processes', 'develop automated migration',
        'maltego', 'parrot security os', 'samurai web',
        'oracle weblogic', 'core banking', 'process-based management',
        'mobile device management', 'interactive media',
        'use an application-specific', 'use computer-aided',
        'use ict ticketing', 'use experience map',
        'use methodologies for user', 'utilise content types',
        'visual presentation', 'web based collaborative',
        'web services', 'online moderation', 'conduct search engine',
        'compose description for web', 'enhance website',
        'study website behaviour', 'create website', 'implement front-end',
        'translate requirements into visual'
    ]

    # --- Cloud & Infrastructure ---
    cloud_keywords = [
        'cloud', 'aws', 'azure', 'gcp', 'serverless', 'saas', 'paas', 'iaas',
        'virtualiz', 'virtualis', 'firewall', 'cybersecurity',
        'encryption', 'linux', 'unix', 'operating system', 'infrastructure',
        'server', 'hardware', 'storage networking', 'backup',
        'disaster recovery', 'domain name', 'dns', 'tcp/ip', 'bandwidth',
        'router', 'switching', 'wan', 'lan', 'telecommunications',
        'ict security', 'cyber', 'access control', 'authentication',
        'malware', 'vulnerability', 'penetration', 'ict risk',
        'ict infrastr', 'ict standard', 'ict performance',
        'ict recovery', 'ict safety', 'ict communic', 'ict power',
        'ict problem', 'ict quality', 'ict process', 'ict software spec',
        'ict market', 'ict sales', 'ict accessib', 'ict environ',
        'itil', 'attack vector', 'ethical hack', 'social engineering test',
        'blackarch', 'metasploit', 'nessus', 'nexpose', 'owasp',
        'thc hydra', 'whitehat', 'wireshark', 'cisco', 'forensic',
        'anti-virus', 'spam protection', 'cryptograph', 'identity management',
        'w3c', 'world wide web consortium',
        'electrical engineer', 'mechatron', 'mechanic', 'battery manag',
        'building systems monitor', 'engineering control', 'engineering principle',
        'engineering process', 'safety engineering', 'state estimation',
        'signal repeat', 'install electronic', 'defence standard',
        'inter-organisational middleware', 'telecom', 'ict legacy',
        'industrial software', 'resource-efficient', 'sustainable technolog',
        'protect ict', 'protect personal data',
        'monitor system performance', 'monitor communication channel',
        'verify formal ict', 'test ict', 'manage email hosting',
        'develop food scanner', 'provide connected car', 'program a cnc',
        'network', 'internet', 'security threat', 'web application security',
        'risk management', 'security engineering', 'internal risk',
        'define security', 'advice on security', 'advise on strength',
        'ensure information security', 'establish an information security',
        'develop information security', 'manage it security',
        'manage system security', 'identify ict system weak',
        'apply information security', 'apply risk management',
        'information security strategy', 'security policies',
        'real-time computing', 'green building', 'lathe', 'tend lathe',
        'set up the controller', 'manufacturing processes',
        'product usage risks', 'embedded systems'
    ]

    for kw in prog_keywords:
        if kw in s:
            return 'Programming Languages'
    for kw in data_keywords:
        if kw in s:
            return 'Databases & Data'
    for kw in tools_keywords:
        if kw in s:
            return 'Frameworks & Tools'
    for kw in cloud_keywords:
        if kw in s:
            return 'Cloud & Infrastructure'

    return 'Soft Skills & Other'


# Apply function to create category column
combined['category'] = combined['preferredLabel'].apply(assign_category_smart)

# Apply immediately
combined = combined[['preferredLabel', 'category', 'source']]
combined.to_csv('outputs/cs_skills_list.csv', index=True)
print("Saved.")

print(combined['category'].value_counts())
print(f"\nTotal: {len(combined)}")


KeyError: "['category'] not in index"

In [ ]:
all_skill_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'skill']
all_occ_nodes   = [n for n, d in G.nodes(data=True) if d['node_type'] == 'occupation']
 
skill_idx = {uri: i for i, uri in enumerate(all_skill_nodes)}
occ_idx   = {uri: i for i, uri in enumerate(all_occ_nodes)}
 
data = HeteroData()
data['skill'].x        = torch.arange(len(all_skill_nodes)).unsqueeze(1).float()
data['occupation'].x   = torch.arange(len(all_occ_nodes)).unsqueeze(1).float()
data['skill'].num_nodes      = len(all_skill_nodes)
data['occupation'].num_nodes = len(all_occ_nodes)
 
essential_edges, optional_edges, broader_edges, narrower_edges = [], [], [], []
 
for _, row in occ_skill_rels.iterrows():
    s, o = row['skillUri'], row['occupationUri']
    if s in skill_idx and o in occ_idx:
        (essential_edges if row['relationType'] == 'essential' else optional_edges)\
            .append((skill_idx[s], occ_idx[o]))
 
for _, row in skill_rels.iterrows():
    s1, s2 = row['originalSkillUri'], row['relatedSkillUri']
    if s1 in skill_idx and s2 in skill_idx:
        (broader_edges if row['relationType'] == 'optional' else narrower_edges)\
            .append((skill_idx[s1], skill_idx[s2]))
 
for _, row in onet_sw_cs.iterrows():
    s_id = f"onet_skill_{row['Element Name']}"
    o_id = f"onet_occ_{row['O*NET-SOC Code']}"
    if s_id in skill_idx and o_id in occ_idx:
        essential_edges.append((skill_idx[s_id], occ_idx[o_id]))
 
def to_edge_index(edges):
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long)
    src, dst = zip(*edges)
    return torch.tensor([list(src), list(dst)], dtype=torch.long)
 
data['skill', 'required_for',  'occupation'].edge_index = to_edge_index(essential_edges)
data['skill', 'optional_for',  'occupation'].edge_index = to_edge_index(optional_edges)
data['skill', 'broader',       'skill'].edge_index      = to_edge_index(broader_edges)
data['skill', 'narrower',      'skill'].edge_index      = to_edge_index(narrower_edges)
 
print("\n=== PyG HeteroData Summary ===")
print(data)
 
transform = RandomLinkSplit(
    num_val=0.15, num_test=0.15, is_undirected=False,
    edge_types=[('skill', 'required_for', 'occupation')],
    rev_edge_types=[None],
)
train_data, val_data, test_data = transform(data)
 
req = data['skill', 'required_for', 'occupation'].edge_index.shape[1]
print(f"\nrequired_for edges total: {req}")
print(f"  Train: {train_data['skill','required_for','occupation'].edge_index.shape[1]}")
print(f"  Val:   {val_data['skill','required_for','occupation'].edge_index.shape[1]}")
print(f"  Test:  {test_data['skill','required_for','occupation'].edge_index.shape[1]}")
import pickle
import os

os.makedirs('outputs', exist_ok=True)

with open('outputs/heterodata_week1.pkl', 'wb') as f:
    pickle.dump({
        'data': data,
        'train': train_data,
        'val': val_data,
        'test': test_data,
        'skill_idx': skill_idx,
        'occ_idx': occ_idx,
        'all_skill_nodes': all_skill_nodes,
        'all_occ_nodes': all_occ_nodes,
    }, f)

print("Saved heterodata_week1.pkl")



=== PyG HeteroData Summary ===
HeteroData(
  skill={
    x=[979, 1],
    num_nodes=979,
  },
  occupation={
    x=[75, 1],
    num_nodes=75,
  },
  (skill, required_for, occupation)={ edge_index=[2, 1797] },
  (skill, optional_for, occupation)={ edge_index=[2, 3093] },
  (skill, broader, skill)={ edge_index=[2, 276] },
  (skill, narrower, skill)={ edge_index=[2, 48] }
)

required_for edges total: 1797
  Train: 1259
  Val:   1259
  Test:  1528
Saved heterodata_week1.pkl


In [ ]:
print(f"CS occ URIs: {len(cs_occ_uris)}")
print(f"CS skill URIs: {len(cs_skill_uris)}")
print(f"Matching occ-skill rows: {len(occ_skill_rels[occ_skill_rels['occupationUri'].isin(cs_occ_uris) & occ_skill_rels['skillUri'].isin(cs_skill_uris)])}")
print(f"Essential only: {len(occ_skill_rels[(occ_skill_rels['occupationUri'].isin(cs_occ_uris)) & (occ_skill_rels['skillUri'].isin(cs_skill_uris)) & (occ_skill_rels['relationType'] == 'essential')])}")

CS occ URIs: 75
CS skill URIs: 881
Matching occ-skill rows: 4890
Essential only: 1797


In [ ]:
# Debug/view a target occupation and its connected skills
if 'G' not in globals():
    raise NameError("G is not defined. Run the graph construction cell first.")

if 'target_occ' not in globals():
    # pick any ESCO occupation node as fallback
    target_occ = next((n for n, d in G.nodes(data=True) if d.get('node_type') == 'occupation'), None)
    if target_occ is None:
        raise NameError("No occupation nodes found in G")

print(G.nodes[target_occ]['node_type'])
print(G.nodes[target_occ]['label'])

# show up to three predecessor skills (if any)
preds = [p for p in G.predecessors(target_occ) if G.nodes[p].get('node_type') == 'skill']
for n in preds[:3]:
    print(G.nodes[n]['node_type'], G.nodes[n]['label'])

occupation
digital games developer
skill digital game genres
skill computer programming
skill 3D texturing


In [ ]:
import pickle

with open('outputs/heterodata_week1.pkl', 'rb') as f:
    saved = pickle.load(f)

print(saved.keys())
print(saved['data'])
print(f"\nSkill nodes: {len(saved['all_skill_nodes'])}")
print(f"Occupation nodes: {len(saved['all_occ_nodes'])}")

dict_keys(['data', 'train', 'val', 'test', 'skill_idx', 'occ_idx', 'all_skill_nodes', 'all_occ_nodes'])
HeteroData(
  skill={
    x=[979, 1],
    num_nodes=979,
  },
  occupation={
    x=[75, 1],
    num_nodes=75,
  },
  (skill, required_for, occupation)={ edge_index=[2, 1797] },
  (skill, optional_for, occupation)={ edge_index=[2, 3093] },
  (skill, broader, skill)={ edge_index=[2, 276] },
  (skill, narrower, skill)={ edge_index=[2, 48] }
)

Skill nodes: 979
Occupation nodes: 75


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pickle
import random

In [ ]:
with open('outputs/heterodata_week1.pkl', 'rb') as f:
    saved = pickle.load(f)

data           = saved['data']
train_data     = saved['train']
val_data       = saved['val']
skill_idx      = saved['skill_idx']
occ_idx        = saved['occ_idx']
all_skill_nodes = saved['all_skill_nodes']
all_occ_nodes   = saved['all_occ_nodes']

In [ ]:
num_skills   = len(all_skill_nodes)   # 979
num_occs     = len(all_occ_nodes)     # 75
num_entities = num_skills + num_occs  # 1054
occ_offset   = num_skills             # occupations start at index 979

RELATIONS = {
    'required_for': 0,
    'optional_for': 1,
    'broader':      2,
    'narrower':     3,
}
num_relations = len(RELATIONS)

In [ ]:
def build_triples(pyg_data):
    triples = []

    # skill → occupation edges
    for rel_name, rel_id in [('required_for', 0), ('optional_for', 1)]:
        edge_key = ('skill', rel_name, 'occupation')
        if edge_key in pyg_data.edge_types:
            ei = pyg_data[edge_key].edge_index
            for i in range(ei.shape[1]):
                h = ei[0, i].item()                  # skill index
                t = ei[1, i].item() + occ_offset     # occupation index (offset)
                triples.append((h, rel_id, t))

    # skill → skill edges
    for rel_name, rel_id in [('broader', 2), ('narrower', 3)]:
        edge_key = ('skill', rel_name, 'skill')
        if edge_key in pyg_data.edge_types:
            ei = pyg_data[edge_key].edge_index
            for i in range(ei.shape[1]):
                h = ei[0, i].item()
                t = ei[1, i].item()
                triples.append((h, rel_id, t))

    return triples

train_triples = build_triples(data)  # use full data for training
val_triples   = build_triples(val_data)

print(f"Total entities:      {num_entities}")
print(f"Total relations:     {num_relations}")
print(f"Training triples:    {len(train_triples)}")
print(f"Validation triples:  {len(val_triples)}")


Total entities:      1054
Total relations:     4
Training triples:    5214
Validation triples:  4676


In [ ]:
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=100, margin=1.0):
        super(TransE, self).__init__()
        self.embedding_dim = embedding_dim
        self.margin        = margin

        # Entity and relation embedding tables
        self.entity_embeddings   = nn.Embedding(num_entities, embedding_dim)
        self.relation_embeddings = nn.Embedding(num_relations, embedding_dim)

        # Initialize embeddings uniformly
        nn.init.uniform_(self.entity_embeddings.weight,
                         -6/np.sqrt(embedding_dim), 6/np.sqrt(embedding_dim))
        nn.init.uniform_(self.relation_embeddings.weight,
                         -6/np.sqrt(embedding_dim), 6/np.sqrt(embedding_dim))

        # Normalize relation embeddings to unit length
        self.relation_embeddings.weight.data = nn.functional.normalize(
            self.relation_embeddings.weight.data, p=2, dim=1
        )

    def score(self, h, r, t):
        # Score = -||h + r - t||_2
        # Lower score = more likely the triple is true
        h_emb = self.entity_embeddings(h)
        r_emb = self.relation_embeddings(r)
        t_emb = self.entity_embeddings(t)
        return -torch.norm(h_emb + r_emb - t_emb, p=2, dim=1)

    def forward(self, pos_h, pos_r, pos_t, neg_h, neg_r, neg_t):
        pos_score = self.score(pos_h, pos_r, pos_t)
        neg_score = self.score(neg_h, neg_r, neg_t)
        # Margin ranking loss
        loss = torch.mean(torch.relu(self.margin - pos_score + neg_score))
        return loss

In [ ]:
def corrupt_triple(h, r, t, num_entities, true_triples_set):
    while True:
        if random.random() < 0.5:
            # Corrupt head
            new_h = random.randint(0, num_entities - 1)
            if (new_h, r, t) not in true_triples_set:
                return (new_h, r, t)
        else:
            # Corrupt tail
            new_t = random.randint(0, num_entities - 1)
            if (h, r, new_t) not in true_triples_set:
                return (h, r, new_t)


In [ ]:
EMBEDDING_DIM = 100
MARGIN        = 1.0
LR            = 0.01
EPOCHS        = 100
BATCH_SIZE    = 128

model     = TransE(num_entities, num_relations, EMBEDDING_DIM, MARGIN)
optimizer = optim.Adam(model.parameters(), lr=LR)

true_triples_set = set(train_triples)
loss_history     = []

print("Starting TransE training...")
print(f"Entities: {num_entities} | Relations: {num_relations} | Triples: {len(train_triples)}")
print("-" * 50)

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_triples)

    epoch_loss = 0.0
    num_batches = 0

    for i in range(0, len(train_triples), BATCH_SIZE):
        batch = train_triples[i:i + BATCH_SIZE]

        # Positive triples
        pos_h = torch.tensor([t[0] for t in batch], dtype=torch.long)
        pos_r = torch.tensor([t[1] for t in batch], dtype=torch.long)
        pos_t = torch.tensor([t[2] for t in batch], dtype=torch.long)

        # Negative triples (one per positive)
        neg_triples = [corrupt_triple(h, r, t, num_entities, true_triples_set)
                       for h, r, t in batch]
        neg_h = torch.tensor([t[0] for t in neg_triples], dtype=torch.long)
        neg_r = torch.tensor([t[1] for t in neg_triples], dtype=torch.long)
        neg_t = torch.tensor([t[2] for t in neg_triples], dtype=torch.long)

        optimizer.zero_grad()
        loss = model(pos_h, pos_r, pos_t, neg_h, neg_r, neg_t)
        loss.backward()
        optimizer.step()

        # Normalize entity embeddings after each update
        model.entity_embeddings.weight.data = nn.functional.normalize(
            model.entity_embeddings.weight.data, p=2, dim=1
        )

        epoch_loss  += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    loss_history.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f}")

print("\nTraining complete.")



Starting TransE training...
Entities: 1054 | Relations: 4 | Triples: 5214
--------------------------------------------------
Epoch  10/100 | Loss: 0.2453
Epoch  20/100 | Loss: 0.2422
Epoch  30/100 | Loss: 0.2373
Epoch  40/100 | Loss: 0.2343
Epoch  50/100 | Loss: 0.2331
Epoch  60/100 | Loss: 0.2308
Epoch  70/100 | Loss: 0.2465
Epoch  80/100 | Loss: 0.2450
Epoch  90/100 | Loss: 0.2299
Epoch 100/100 | Loss: 0.2353

Training complete.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), loss_history, color='#4A90D9', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('TransE Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/transe_loss_curve.png', dpi=150)
plt.show()
print("Loss curve saved.")

Loss curve saved.


C:\Users\kanis\AppData\Local\Temp\ipykernel_11208\904926632.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'embedding_dim':    EMBEDDING_DIM,
    'num_entities':     num_entities,
    'num_relations':    num_relations,
    'occ_offset':       occ_offset,
    'loss_history':     loss_history,
}, 'outputs/transe_checkpoint.pt')
print("Checkpoint saved to outputs/transe_checkpoint.pt")

Checkpoint saved to outputs/transe_checkpoint.pt


In [ ]:
def evaluate(model, test_triples, num_entities, occ_offset, batch_size=64):
    model.eval()
    
    ranks = []
    
    with torch.no_grad():
        for idx, (h, r, t) in enumerate(test_triples):
            # Only evaluate skill → occupation triples (required_for)
            if r != 0:
                continue

            h_tensor = torch.tensor([h], dtype=torch.long)
            r_tensor = torch.tensor([r], dtype=torch.long)

            # Score against ALL occupation candidates
            all_tails = torch.arange(occ_offset, occ_offset + 75, dtype=torch.long)
            h_exp     = h_tensor.expand(len(all_tails))
            r_exp     = r_tensor.expand(len(all_tails))

            scores    = model.score(h_exp, r_exp, all_tails)
            true_score = model.score(h_tensor, r_tensor,
                                     torch.tensor([t], dtype=torch.long))

            # Rank of true tail (higher score = better)
            rank = (scores >= true_score).sum().item()
            ranks.append(rank)

    ranks   = np.array(ranks)
    mrr     = np.mean(1.0 / ranks)
    hits_1  = np.mean(ranks <= 1)
    hits_3  = np.mean(ranks <= 3)
    hits_10 = np.mean(ranks <= 10)

    return mrr, hits_1, hits_3, hits_10

In [ ]:
test_triples = build_triples(saved['test'])
print(f"Test triples: {len(test_triples)}")

mrr, h1, h3, h10 = evaluate(model, test_triples, num_entities, occ_offset)

print("\n=== TransE Baseline Results ===")
print(f"MRR:     {mrr:.4f}")
print(f"Hits@1:  {h1:.4f}")
print(f"Hits@3:  {h3:.4f}")
print(f"Hits@10: {h10:.4f}")

# Save results to CSV
import pandas as pd
results_df = pd.DataFrame([{
    'model': 'TransE',
    'MRR': round(mrr, 4),
    'Hits@1': round(h1, 4),
    'Hits@3': round(h3, 4),
    'Hits@10': round(h10, 4)
}])
print("\n")
print(results_df.to_string(index=False))
results_df.to_csv('outputs/baseline_results.csv', index=False)
print("\nSaved to outputs/baseline_results.csv")

Test triples: 4945

=== TransE Baseline Results ===
MRR:     0.3248
Hits@1:  0.1741
Hits@3:  0.3704
Hits@10: 0.6538


 model    MRR  Hits@1  Hits@3  Hits@10
TransE 0.3248  0.1741  0.3704   0.6538

Saved to outputs/baseline_results.csv
